# Download các thư viện cần thiết

In [1]:
import os
import polars as pl
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

# Tạo hàm để đọc file parquet (đọc các file parquet - data sau khi được processed)

In [6]:
def read_parquet_user(train_path: str):
    # Lấy tất cả các file parquet trong thư mục
    files = [os.path.join(train_path, f) for f in os.listdir(train_path) if f.endswith('.parquet')]
    
    # Phân loại các file theo loại tên
    user_chunk_files = [file for file in files if 'user_chunk' in file]
        
    # Đọc các file riêng biệt thành DataFrame
    user_chunk_df = pl.concat([pl.read_parquet(file) for file in user_chunk_files]) if user_chunk_files else None
        
    # Trả về một dictionary chứa các DataFrame
    return user_chunk_df

In [7]:
def read_parquet_item(train_path: str):
    # Lấy tất cả các file parquet trong thư mục
    files = [os.path.join(train_path, f) for f in os.listdir(train_path) if f.endswith('.parquet')]
    
    # Phân loại các file theo loại tên
    item_chunk_files = [file for file in files if 'item_chunk' in file]
        
    # Đọc các file riêng biệt thành DataFrame
    item_chunk_df = pl.concat([pl.read_parquet(file) for file in item_chunk_files]) if item_chunk_files else None
        
    # Trả về một dictionary chứa các DataFrame
    return item_chunk_df

In [8]:
def read_parquet_transaction(train_path: str):
    # Lấy tất cả các file parquet trong thư mục
    files = [os.path.join(train_path, f) for f in os.listdir(train_path) if f.endswith('.parquet')]
    
    # Phân loại các file theo loại tên
    purchase_chunk_files = [file for file in files if 'purchase_history_daily_chunk' in file]
        
    # Đọc các file riêng biệt thành DataFrame
    purchase_chunk_df = pl.concat([pl.read_parquet(file) for file in purchase_chunk_files]) if purchase_chunk_files else None
        
    # Trả về một dictionary chứa các DataFrame
    return purchase_chunk_df

# Tạo hàm lưu file parquet sau mỗi task

In [9]:
def split_and_save_parquet(df, num_files, output_dir):
    """
    Tách DataFrame thành nhiều file Parquet và lưu vào thư mục đích.
    
    :param df: DataFrame cần tách
    :param num_files: Số lượng file Parquet muốn tách
    :param output_dir: Thư mục lưu các file Parquet
    """
    # Đảm bảo thư mục tồn tại
    os.makedirs(output_dir, exist_ok=True)
    
    # Tính số dòng mỗi file sẽ có
    num_rows = df.height
    rows_per_file = num_rows // num_files

    # Tách DataFrame thành các phần và lưu mỗi phần vào một file Parquet
    for i in range(num_files):
        start_row = i * rows_per_file
        # Đảm bảo phần cuối cùng sẽ chứa tất cả các dòng còn lại
        end_row = (i + 1) * rows_per_file if i < num_files - 1 else num_rows
        
        # Tách phần DataFrame
        split_df = df[start_row:end_row]
        
        # Lưu phần DataFrame vào file .parquet
        file_path = os.path.join(output_dir, f"item_age_new.parquet")
        split_df.write_parquet(file_path)
        print(f"Đã lưu file: {file_path}")

# Các dặc trưng mới

In [17]:
item_age = pl.read_parquet("/datastore/uittogether/LuuTru/Thanhld/CS116-DoAn/Phase-2/Pipeline-training3/Thanh/data/new-feature/item_age.parquet")

In [19]:
item_age['item_id', 'age_min_month', 'age_max_month'].head()

item_id,age_min_month,age_max_month
str,i64,i64
"""0502020000004""",9,60
"""0010290040150""",36,60
"""0008010000015""",0,12
"""0020010000094""",3,36
"""0020010000098""",12,36


In [21]:
split_and_save_parquet(item_age['item_id', 'age_min_month', 'age_max_month'], 1, "/datastore/uittogether/LuuTru/Thanhld/CS116-DoAn/Phase-2/Pipeline-training3/Thanh/data/new-feature")

Đã lưu file: /datastore/uittogether/LuuTru/Thanhld/CS116-DoAn/Phase-2/Pipeline-training3/Thanh/data/new-feature/item_age_new.parquet


In [6]:
brand_segment_df = pl.read_parquet("./new-feature/brand_segment.parquet")
brand_segment_df.head()

category_l1,brand_segment
str,i64
"""Sữa""",2
"""Sữa""",1
"""Sữa""",2
"""Sữa""",0
"""Sữa""",0


In [16]:
customer_age_features_df = pl.read_parquet("/datastore/uittogether/LuuTru/Thanhld/CS116-DoAn/Phase-2/Pipeline-training3/Thanh/data/new-feature/customer_age_features_2401_2501.parquet")
customer_age_features_df = customer_age_features_df['customer_id', 'age_final']
customer_age_features_df

customer_id,age_final
i32,f64
4748251,null
6704093,13.6
8092223,null
7105333,null
2979082,null
…,…
6878538,null
7844605,null
114947,null


In [8]:
customer_behavior_df = pl.read_parquet("./new-feature/customer_behavior.parquet")
# customer_behavior_df = customer_behavior_df['customer_id', 'buy_segment']
customer_behavior_df

customer_id,total_unique_categories,total_days,avg_categories_per_day,buy_segment_raw,buy_segment
i32,u32,u32,f64,i32,i32
7912392,1,1,1.0,0,0
7396997,1,1,1.0,0,0
7279939,40,25,1.6,2,1
690290,2,2,1.0,0,0
6323569,1,1,1.0,0,0
…,…,…,…,…,…
1599282,1,1,1.0,0,0
7105696,13,10,1.3,0,0
7413857,1,1,1.0,0,0


In [20]:
customer_luxury_df = pl.read_parquet("./new-feature/customer_luxury.parquet")
customer_luxury_df = customer_luxury_df['customer_id', 'luxury_level']
customer_luxury_df.head()

customer_id,luxury_level
i32,i32
3915079,0
335117,0
703093,1
4200016,1
2524669,1


In [10]:
price_segment_df = pl.read_parquet("./new-feature/price_segment.parquet")
price_segment_df = price_segment_df['item_id', 'price_segment']
price_segment_df.head()

item_id,price_segment
str,i64
"""0502020000004""",0
"""0010290040150""",0
"""0008010000015""",0
"""0020010000094""",1
"""0020010000098""",1


In [11]:
top10_by_cat_month_df = pl.read_parquet("./new-feature/top10_by_cat_month.parquet")
# top10_by_cat_month_df = top10_by_cat_month_df['item_id', 'rank']
top10_by_cat_month_df.head()

month,category_l1,item_id,total_sold,rank
str,str,str,i32,u32
"""2024-01""","""Babycare""","""5950000000001""",17358,1
"""2024-01""","""Babycare""","""0007160000082""",10646,2
"""2024-01""","""Babycare""","""0007150000144""",10069,3
"""2024-01""","""Babycare""","""0007150000031""",8871,4
"""2024-01""","""Babycare""","""0203000000004""",8475,5


# Đọc data

In [2]:
df = pl.read_parquet("/datastore/uittogether/LuuTru/Thanhld/CS116-DoAn/Phase-2/Pipeline-training3/Thanh2/artifacts/stage2_train_new_item_rec.parquet")

In [5]:
df.head()

customer_id,item_id,stage1_rank,stage1_score,sim_max,sim_avg,brand_match_cnt,cat2_match_cnt,feat_days_since_cat,feat_brand_affinity,feat_price_ratio,feat_pop_30d,feat_log_price,feat_baby_age,feat_is_age_match,feat_cold_pop_score,feat_prev_month_cold_score,support_cnt,label
str,str,i64,f64,f64,f64,i64,i64,i64,f64,f64,i64,f64,f64,i64,f64,f64,i64,i64
"""8211116""","""1215000000002""",1,0.0,0.0,0.0,0,0,999,0.0,1.0,1436,10.434145,3.5,0,0.873513,0.034743,0,0
"""8211116""","""4690000000001""",2,0.0,0.0,0.0,0,0,999,0.0,1.0,39630,10.463132,3.5,0,0.733226,1.0,0,1
"""8211116""","""6768000000005""",3,0.0,0.0,0.0,0,0,999,0.0,1.0,16525,12.751303,3.5,0,0.492953,0.115263,0,0
"""8211116""","""6768000000004""",4,0.0,0.0,0.0,0,0,999,0.0,1.0,14442,12.751303,3.5,0,0.409705,0.109472,0,0
"""8211116""","""6768000000003""",5,0.0,0.0,0.0,0,0,999,0.0,1.0,10453,12.751303,3.5,1,0.303847,0.104774,0,0
